# Compartment - mmBERT pipeline (Kaggle)

Chạy toàn bộ M3 chain trong một session: **probe → warmup → train5 → predict**.

### Prerequisites
- Repo được mount như Kaggle dataset (folder `Compartment`) — re-upload bản code **mới nhất** (M3) trước khi chạy, dữ liệu cũ không có `mm/`.
- Path repo được dò tự động trong `/kaggle/input/**/Compartment`.

### Knobs (sửa dòng `MODE` bên dưới)
- `MODE = 'all'` chạy cả chain; đặt `probe | warmup | train5 | predict` để chạy riêng.
- `WARMUP_EPOCHS`, `WARMUP_BATCH` (hạ xuống 16 nếu OOM ở phase MLM vì logits vocab khổng lồ).
- `MLM_DATA`: các file context dùng cho MLM warmup.
- `TRAIN5_SET`: thêm `--set` cho train5 (cách nhau bằng dấu phẩy).

Outputs nằm ở `/kaggle/working`: `models/`, `submission/`, `metrics.json`, `history.json`, `oof_predictions.npz`, `trial_metrics.json`.

In [ ]:
import glob, os, subprocess, sys
from pathlib import Path

def _importable(name: str) -> bool:
    try:
        __import__(name)
        return True
    except Exception:
        return False

# ---- locate the repository ----------------------------------------------
cands = [Path('/kaggle/input/datasets/ieltsmater/compartment/Compartment')]
cands += [Path(p) for p in glob.glob('/kaggle/input/**/Compartment', recursive=True)]
cands += [Path('/kaggle/working/Compartment')]
REPO = next((p for p in cands if (p / 'run.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Compartment repo not found under /kaggle/input or /kaggle/working')
os.chdir(REPO)
print('repo:', REPO)

# ---- knobs --------------------------------------------------------------
MODE = os.environ.get('MODE', 'all')            # all | probe | warmup | train5 | predict
WARMUP_EPOCHS = int(os.environ.get('WARMUP_EPOCHS', '6'))
WARMUP_BATCH = os.environ.get('WARMUP_BATCH', '')   # e.g. 16 if OOM in MLM phase
MLM_DATA = 'en-nn-train.tsv,de-nn-train.tsv,en-pv-train.tsv,de-pv-train.tsv,nctti_en.tsv'
TRAIN5_SET = os.environ.get('TRAIN5_SET', '')       # extra --set flags, comma-separated

# ---- ensure imports exist (Kaggle already ships them) -------------------
for m in ('torch', 'transformers', 'pandas', 'numpy', 'sklearn', 'scipy'):
    if not _importable(m):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', m], check=False)

def run(args):
    print('\n>>>', ' '.join(args))
    subprocess.run(args, cwd=REPO, check=True)

In [ ]:
# --- M2 gate: real-tokenizer alignment + MLM masking sanity ---------------
if MODE in ('all', 'probe'):
    run([sys.executable, 'run.py', 'probe'])

In [ ]:
# --- Phase 0: compound-aware MLM warmup + LoRA merge ----------------------
# Writes /kaggle/working/models/warmup_merged.pt (used as base by train5).
if MODE in ('all', 'warmup'):
    args = [sys.executable, 'run.py', 'warmup',
            '--set', f'warmup_mlm_epochs={WARMUP_EPOCHS}',
            '--set', f'mlm_data_paths={MLM_DATA}']
    if WARMUP_BATCH:
        args += ['--set', f'warmup_batch_size={WARMUP_BATCH}']
    run(args)

In [ ]:
# --- Phase 1: 5-fold CV (compound-grouped, no leak) + OOF -----------------
if MODE in ('all', 'train5'):
    args = [sys.executable, 'run.py', 'train5']
    if TRAIN5_SET:
        args += [x.strip() for x in TRAIN5_SET.split(',') if x.strip()]
    run(args)

In [ ]:
# --- Predict trial (5-fold ensemble) + submission -------------------------
if MODE in ('all', 'predict'):
    run([sys.executable, 'run.py', 'predict'])

In [ ]:
import json
import pandas as pd

work = Path('/kaggle/working')
for name in ('metrics.json', 'trial_metrics.json'):
    p = work / name
    if p.is_file():
        print(f'--- {name} ---')
        print(json.dumps(json.loads(p.read_text(encoding='utf-8')), indent=2))

sub = work / 'submission' / 'en-nn-trial-pred.tsv'
if sub.is_file():
    df = pd.read_csv(sub, sep='\t', header=None, names=['tID', 'Modifier', 'Head'])
    print('\n--- submission (en-nn-trial-pred.tsv, no header) ---')
    print(df.head())
    print('rows:', len(df))

### Nộp submission
1. File `en-nn-trial-pred.tsv` nằm ở `/kaggle/working/submission/`.
2. Download rồi nộp ở challenge (3 cột `tID, Modifier, Head`, không header).

### Lưu ý
- **trial ρ là artifact n=2** — đừng dùng trial rho làm tín hiệu; xem **OOF ρ** (mean của `Mod`/`Head`) trong `metrics.json`.
- Muốn chạy lại chain với config khác: chỉnh `TRAIN5_SET`, ví dụ `TRAIN5_SET='--set,batch_size=16,--set,freeze_epochs=2'`. WARMUP thay đổi thì phải chạy lại `warmup` + `train5` + `predict` trong cùng session (ckpt ở `/kaggle/working` không tồn tại giữa các session).